# Week 6: A Neural Network Recognizes Handwritten Digits

[Open in Colab](https://colab.research.google.com/github/astarryknight/ai-ml/blob/main/notebooks/week-06-digit-classifier.ipynb)

**Estimated time:** 75–90 minutes  
**Finish line:** train one small neural network, compare it with a linear baseline, and inspect images it gets wrong.

The library handles backpropagation. We focus on the inputs, the prediction, and whether the mistakes make sense.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

## 2. Look at the images before modeling

Each example is an 8 by 8 grayscale image. The model receives 64 pixel values and predicts one of ten digit labels.

In [ ]:
digits = load_digits()
images = digits.images
X = digits.data / 16.0
y = digits.target

fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), images[:12], y[:12]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Images:", images.shape)
print("Model inputs:", X.shape)

## 3. Split the examples

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))

## 4. Train a linear baseline

The baseline creates a linear score for each digit. The neural network should earn its extra complexity.

In [ ]:
baseline = LogisticRegression(max_iter=2000, random_state=42)
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print(f"Linear baseline accuracy: {baseline_accuracy:.3f}")

## 5. Train one small neural network

The architecture has 64 inputs, one hidden layer with 32 units, and ten possible outputs. The model code remains short.

In [ ]:
network = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.003,
    max_iter=500,
    n_iter_no_change=20,
    random_state=42,
)

network.fit(X_train, y_train)
network_predictions = network.predict(X_test)
network_accuracy = accuracy_score(y_test, network_predictions)

print(f"Neural-network accuracy: {network_accuracy:.3f}")
print(f"Training epochs: {network.n_iter_}")

## 6. Inspect the learning curve

Loss measures how far the model's outputs are from the correct answers during training. The overall direction matters more than a perfectly smooth line.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(network.loss_curve_, color="#147D88")
plt.title("Training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## 7. Compare the models

In [ ]:
comparison = pd.DataFrame({
    "model": ["Linear baseline", "Neural network"],
    "test_accuracy": [baseline_accuracy, network_accuracy],
})
comparison

## 8. Inspect the confusion matrix

Values on the diagonal are correct predictions. Values away from the diagonal show which digits the model confuses.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, network_predictions, cmap="Blues", colorbar=False
)
plt.title("Neural-network confusion matrix")
plt.show()

## 9. Look at actual mistakes

Images reveal details that a single accuracy number cannot show.

In [ ]:
wrong = np.flatnonzero(network_predictions != y_test)
count = min(12, len(wrong))

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax in axes.ravel():
    ax.axis("off")
for ax, test_index in zip(axes.ravel(), wrong[:count]):
    ax.imshow(X_test[test_index].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"True: {y_test[test_index]}\nPred: {network_predictions[test_index]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Your experiment

Change only the hidden-layer size. Try 16, 32, and 64 units. Record the test accuracy, training epochs, and one pair of digits the model confuses.

| Hidden units | Epochs | Test accuracy | Notable confusion |
| ---: | ---: | ---: | --- |
| 16 |  |  |  |
| 32 |  |  |  |
| 64 |  |  |  |

## 11. Validation

In [ ]:
assert images.shape[1:] == (8, 8)
assert X.shape[1] == 64
assert len(network.loss_curve_) > 1
assert len(network_predictions) == len(y_test)
assert 0 <= network_accuracy <= 1
print("Validation passed: the model trained and produced test predictions.")

## Quick self-check

1. Why does one image become 64 model inputs?
2. What does the hidden layer add that the linear baseline does not?
3. Why should we inspect incorrect images as well as accuracy?

## Finish line

- Baseline and neural-network accuracy
- One training-loss plot
- One confusion matrix
- At least two inspected mistakes
- A three-row hidden-layer experiment